# Kabaddi Coach Dashboard v10

**Full browser UI - 5 ML models - H2H report - Save to all 3 CSVs**

**v10 Bug Fix:** Substitution dropdowns now correctly show only the selected team's players.
All player IDs normalized to `Number()` in `buildGrid`, `getPids`, `toggleP`, `refreshSubs`, and `applySub`
to prevent strict-equality (`===`) mismatches between JSON-returned string IDs and numeric IDs in the `PR` dict.

1. Edit `DATA_DIR` in Cell 1 to point to your PKL CSV folder
2. Run All Cells
3. Click the link that appears after Cell 5


---
## Cell 1 - Imports and Config

In [1]:
# CELL 1 - IMPORTS & CONFIG
import os, math, json, copy, warnings, threading, webbrowser, time
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes     import GaussianNB
from sklearn.svm             import SVC
from flask import Flask, request, jsonify, render_template_string
from IPython.display import display, HTML

# EDIT THESE
DATA_DIR   = os.path.expanduser('~/Downloads')
OUTPUT_DIR = os.path.expanduser('~/Downloads')
PORT       = 5055

MODEL_PATH = os.path.join(OUTPUT_DIR, 'kabaddi_model_v7.joblib')
os.makedirs(OUTPUT_DIR, exist_ok=True)
# 4 features only — score_diff is NOT in model features
# Score impact is applied directly as a sigmoid shift after prediction
# (same approach as the original working Kabaddi_.ipynb notebook)
FEATURES = ['raid_success_A','tackle_success_A','raid_success_B','tackle_success_B']

print('Config ready')
print(f'  Data   : {DATA_DIR}')
print(f'  Model  : {MODEL_PATH}')
print(f'  Server : http://127.0.0.1:{PORT}')


Config ready
  Data   : C:\Users\venug/Downloads
  Model  : C:\Users\venug/Downloads\kabaddi_model_v7.joblib
  Server : http://127.0.0.1:5055


---
## Cell 2 - Load Data

In [2]:
# CELL 2 - LOAD DATA
def _load(fname):
    df = pd.read_csv(os.path.join(DATA_DIR, fname))
    df.columns = df.columns.str.strip()
    return df

players = _load('PKL_players.csv')
teams   = _load('PKL_teams.csv')
events  = _load('PKL_events.csv')
matches = _load('PKL_matches.csv')

for name, df in [('Players',players),('Teams',teams),('Events',events),('Matches',matches)]:
    print(f'  {name:10s}: {df.shape[0]:>5} rows x {df.shape[1]:>3} cols')

# Latest-season roster filter (fixes huge player list)
_lat = players.groupby('team_id')['match_id'].max().reset_index()
_lat.columns = ['team_id','latest_match_id']
players_latest = (players.merge(_lat, on='team_id')
                  .query('match_id == latest_match_id').copy())

print(f'\n  Latest roster: {players_latest["player_id"].nunique()} players '
      f'across {players_latest["team_id"].nunique()} teams '
      f'(~{round(players_latest["player_id"].nunique()/players_latest["team_id"].nunique())} per team)')
print('Data loaded OK')


  Players   : 14887 rows x  29 cols
  Teams     :  1218 rows x  34 cols
  Events    : 59362 rows x  39 cols
  Matches   :   610 rows x  17 cols

  Latest roster: 120 players across 12 teams (~10 per team)
Data loaded OK


---
## Cell 3 - Feature Engineering

In [3]:
# CELL 3 - FEATURE ENGINEERING
for col in ['stats.raids.successful','stats.raids.total','stats.tackles.successful','stats.tackles.total']:
    if col not in teams.columns: teams[col] = 0

teams['raid_success']   = np.where(teams['stats.raids.total']>0,
    teams['stats.raids.successful']/teams['stats.raids.total'], 0.0)
teams['tackle_success'] = np.where(teams['stats.tackles.total']>0,
    teams['stats.tackles.successful']/teams['stats.tackles.total'], 0.0)

if 'match_id' not in teams.columns: teams['match_id'] = range(len(teams))
teams = teams.sort_values(['id','match_id']).reset_index(drop=True)
tp = teams[['match_id','id','raid_success','tackle_success']].rename(columns={'id':'team_id'}).copy()
tp['raid_roll5']   = tp.groupby('team_id')['raid_success'].shift(1).rolling(5,min_periods=1).mean()
tp['tackle_roll5'] = tp.groupby('team_id')['tackle_success'].shift(1).rolling(5,min_periods=1).mean()

rows = []
for mid, grp in tp.groupby('match_id'):
    if grp.shape[0] != 2: continue
    a, b = grp.iloc[0], grp.iloc[1]
    rows.append(dict(match_id=mid,
        teamA_id=int(a['team_id']), teamB_id=int(b['team_id']),
        raid_success_A=a['raid_roll5'], raid_success_B=b['raid_roll5'],
        tackle_success_A=a['tackle_roll5'], tackle_success_B=b['tackle_roll5']))
match_df = pd.DataFrame(rows)

def _parse_winner(r):
    if isinstance(r, str) and ' beat ' in r: return r.split(' beat ')[0].strip()
    return None

if 'winner_team_id' in matches.columns:
    match_df = match_df.merge(matches[['match_id','winner_team_id']], on='match_id', how='left')
    match_df['winA'] = (match_df['winner_team_id'] == match_df['teamA_id']).astype(int)
elif 'result' in matches.columns:
    matches['_wname'] = matches['result'].apply(_parse_winner)
    n2id = teams[['id','name']].drop_duplicates().set_index('name')['id'].to_dict()
    matches['_wid'] = matches['_wname'].map(n2id)
    match_df = match_df.merge(matches[['match_id','_wid']], on='match_id', how='left')
    match_df['winA'] = (match_df['_wid'] == match_df['teamA_id']).astype(int)
else:
    np.random.seed(42); match_df['winA'] = np.random.randint(0,2,len(match_df))

# No score_diff augmentation needed — score impact is applied
# as a direct sigmoid shift in predict_win(), not through the model.
# This is exactly how the original Kabaddi_.ipynb worked.
match_df_aug = match_df  # use base match data directly

player_dist = {}
for pid, grp in players.groupby('player_id'):
    pts = grp['player_total_points'].dropna() if 'player_total_points' in grp.columns else pd.Series(dtype=float)
    if len(pts) > 2:
        player_dist[int(pid)] = {'mu': float(pts.mean()), 'std': float(max(pts.std(),0.1))}

team_latest_stats = {}
for tid in teams['id'].unique():
    t_rows = tp[tp['team_id']==tid].sort_values('match_id')
    last = t_rows.iloc[-1] if not t_rows.empty else None
    team_latest_stats[int(tid)] = {
        'raid_success'  : float(last['raid_roll5'])   if last is not None and pd.notna(last['raid_roll5'])   else 0.50,
        'tackle_success': float(last['tackle_roll5']) if last is not None and pd.notna(last['tackle_roll5']) else 0.40,
    }

team_avg_mu = {}
for tid in teams['id'].unique():
    pids = players[players['team_id']==tid]['player_id'].unique()
    mus  = [player_dist.get(int(p),{}).get('mu',0) for p in pids]
    team_avg_mu[int(tid)] = float(np.mean(mus)) if mus else 1.0

print('Feature engineering done')
print(f'  Aug rows: {len(match_df_aug)} | Player profiles: {len(player_dist)} | Team caches: {len(team_latest_stats)}')


ValueError: cannot convert float NaN to integer

---
## Cell 4 - Train 5 Models (best ROC-AUC wins)

In [4]:
# CELL 4 - MODEL TRAINING
# Training on 4 clean features (raid/tackle success rates) — no score_diff.
# Score impact is handled separately via direct sigmoid shift in predict_win().
# This matches the approach of the original working Kabaddi_.ipynb:
#   - Model learns team quality from historical raid/tackle rates
#   - Live score is layered on top independently, always working

dfm = match_df_aug.dropna(subset=['winA']).copy()
X   = dfm[FEATURES].fillna(0.5)
y   = dfm['winA'].astype(float).astype(int)

if len(X) < 10:
    raise ValueError("Too few labelled matches. Check winner column in data.")

print(f"  Training rows : {len(X)}")
print(f"  Win-rate      : {y.mean():.1%}")
print(f"  Features      : {FEATURES}")
print()

n_splits = max(2, min(5, int(y.value_counts().min())))
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
candidates = {}

def _try(name, clf):
    try:
        s = cross_val_score(clf, X, y, cv=cv, scoring='roc_auc').mean()
        candidates[name] = (clf, round(s, 4))
        print(f"  {name:35s}  ROC-AUC = {s:.4f}")
    except Exception as e:
        print(f"  {name:35s}  FAILED: {e}")

_try('LogisticRegression',            LogisticRegression(max_iter=5000))
_try('RandomForest (300 trees)',       RandomForestClassifier(n_estimators=300, random_state=42))
_try('GradientBoosting (200 trees)',  GradientBoostingClassifier(n_estimators=200, random_state=42))
_try('GaussianNaiveBayes',            GaussianNB())
_try('SVM-RBF (calibrated)',          SVC(kernel='rbf', probability=True, C=1.0))

if not candidates:
    raise RuntimeError("All models failed.")

# Select best model by ROC-AUC
best_name, (best_clf, best_score) = max(candidates.items(), key=lambda kv: kv[1][1])
best_clf.fit(X, y)
joblib.dump(best_clf, MODEL_PATH)
model = best_clf

print(f"\n  Best model: {best_name}  (ROC-AUC = {best_score})")
print("  All models are valid here — no score_diff in features so no overfitting risk")

# Show feature weights if logistic
if hasattr(model, 'coef_'):
    print("\n  Feature weights:")
    for f, c in zip(FEATURES, model.coef_[0]):
        print(f"    {f:25s}: {c:+.3f}")

# Verify score shift works independently
import pandas as _pd
_f = dict(zip(FEATURES, [0.52, 0.45, 0.52, 0.45]))
_bp = model.predict_proba(_pd.DataFrame([_f], columns=FEATURES))[0, 1]
print(f"\n  Base prob (balanced teams): {_bp:.3f}")
print("  Score shift (applied in predict_win, not model):")
for sd in [0, 5, 10, -5, -10]:
    _shift = (1.0 / (1.0 + math.exp(-sd / 4.0)) - 0.5) * 0.5
    _p = max(0.02, min(0.98, _bp + _shift))
    print(f"    score_diff={sd:+3d}  ->  {_p:.1%}")

print("\nModel trained and saved")


  Training rows : 609
  Win-rate      : 42.9%
  Features      : ['raid_success_A', 'tackle_success_A', 'raid_success_B', 'tackle_success_B']

  LogisticRegression                   ROC-AUC = 0.6082
  RandomForest (300 trees)             ROC-AUC = 0.5893
  GradientBoosting (200 trees)         ROC-AUC = 0.5842
  GaussianNaiveBayes                   ROC-AUC = 0.6052
  SVM-RBF (calibrated)                 ROC-AUC = 0.5910

  Best model: LogisticRegression  (ROC-AUC = 0.6082)
  All models are valid here — no score_diff in features so no overfitting risk

  Feature weights:
    raid_success_A           : +1.387
    tackle_success_A         : +1.570
    raid_success_B           : -0.479
    tackle_success_B         : -1.361

  Base prob (balanced teams): 0.473
  Score shift (applied in predict_win, not model):
    score_diff= +0  ->  47.3%
    score_diff= +5  ->  61.2%
    score_diff=+10  ->  68.5%
    score_diff= -5  ->  33.4%
    score_diff=-10  ->  26.1%

Model trained and saved


---
## Cell 5 - Flask Web Server

Run once. A clickable link appears below. The server runs in a background thread.

In [5]:
# CELL 5 - FLASK WEB SERVER
import copy, datetime

# --- Prediction engine ---

def lineup_strength(pids):
    mus = [player_dist.get(int(p), {}).get('mu', 0) for p in pids]
    return float(np.mean(mus)) if mus else 0.0

def compute_features(state):
    tidA = state['teamA_id']; tidB = state['teamB_id']
    sA = team_latest_stats.get(tidA, {'raid_success':0.50,'tackle_success':0.40})
    sB = team_latest_stats.get(tidB, {'raid_success':0.50,'tackle_success':0.40})
    avgA = team_avg_mu.get(tidA, 1.0) or 1.0
    avgB = team_avg_mu.get(tidB, 1.0) or 1.0
    selA = lineup_strength(state['starters_A'])
    selB = lineup_strength(state['starters_B'])
    adjA = (selA - avgA) / (avgA + 1e-6)
    adjB = (selB - avgB) / (avgB + 1e-6)
    state['features'] = {
        'raid_success_A'  : float(np.clip(sA['raid_success']   * (1 + 0.15 * adjA), 0.25, 0.80)),
        'tackle_success_A': float(np.clip(sA['tackle_success'] * (1 + 0.12 * adjA), 0.25, 0.80)),
        'raid_success_B'  : float(np.clip(sB['raid_success']   * (1 + 0.15 * adjB), 0.25, 0.80)),
        'tackle_success_B': float(np.clip(sB['tackle_success'] * (1 + 0.12 * adjB), 0.25, 0.80)),
        # score_diff stored separately — NOT fed into model
        'score_diff'      : float(state['scoreA'] - state['scoreB']),
    }
    return state['features']

def predict_win(state):
    """
    Two-layer probability:
      Layer 1 — ML model on 4 raid/tackle features → base probability
      Layer 2 — Direct sigmoid shift for live score → final probability

    Layer 2 uses: shift = (sigmoid(-score_diff/4) - 0.5) * 0.5
    This is identical to the original working Kabaddi_.ipynb approach.
    It ALWAYS moves probability regardless of model training, guaranteeing
    visible response to every score change (even 1-point differences).

    At score_diff=+1:  +3.1% shift
    At score_diff=+5:  +13.9% shift
    At score_diff=+10: +21.2% shift
    """
    if not state.get('features'):
        compute_features(state)

    # Layer 1: model prediction on 4 clean features (no score_diff)
    model_features = {k: v for k, v in state['features'].items() if k in FEATURES}
    fv     = pd.DataFrame([model_features], columns=FEATURES)
    base_p = float(model.predict_proba(fv)[0, 1])

    # Layer 2: direct sigmoid score shift (same formula as Kabaddi_.ipynb)
    sd     = float(state.get('scoreA', 0)) - float(state.get('scoreB', 0))
    shift  = (1.0 / (1.0 + math.exp(-sd / 4.0)) - 0.5) * 0.5
    p      = float(np.clip(base_p + shift, 0.02, 0.98))

    state['probA'] = p
    return p

def bayes_update(prior, pid, observed, alpha=0.4):
    pid = int(pid)
    if pid not in player_dist: return prior
    mu, std = player_dist[pid]['mu'], player_dist[pid]['std']
    if std <= 0: return prior
    LR   = math.exp(alpha * (observed - mu) / std)
    LR   = max(1e-3, min(1e3, LR))
    odds = (prior / (1.0 - prior + 1e-12)) * LR
    return float(np.clip(odds / (1.0+odds), 0.0, 1.0))

def simulate_sub(state, side, out_pid, in_pid):
    s2 = copy.deepcopy(state)
    sk, bk = 'starters_'+side, 'bench_'+side
    if out_pid in s2[sk]:    s2[sk].remove(out_pid)
    if in_pid not in s2[sk]: s2[sk].append(in_pid)
    if in_pid  in s2[bk]:   s2[bk].remove(in_pid)
    if out_pid not in s2[bk]: s2[bk].append(out_pid)
    compute_features(s2)
    return float(model.predict_proba(pd.DataFrame([s2['features']], columns=FEATURES))[0, 1])

def roster_for_team(tid):
    p = players_latest[players_latest['team_id'] == tid].copy()
    if p.empty: p = players[players['team_id'] == tid].copy()
    if p.empty: return [], [], []
    avg_pts = players.groupby('player_id')['player_total_points'].mean().to_dict()
    all_ids = sorted(p['player_id'].unique().tolist(), key=lambda x: avg_pts.get(x, 0), reverse=True)
    if 'player_on_starter' in p.columns:
        st = [pid for pid in all_ids
              if p.loc[p['player_id']==pid, 'player_on_starter'].iloc[0] == True]
        bn = [pid for pid in all_ids if pid not in st]
        if len(st) >= 5: return st[:7], bn[:5], all_ids
    return all_ids[:7], all_ids[7:], all_ids

def get_player_info(pid):
    pid = int(pid)
    rows = players[players['player_id'] == pid]
    name = str(rows['player_name'].dropna().iloc[0]) if not rows.empty and 'player_name' in rows.columns else f'Player-{pid}'
    d = player_dist.get(pid, {})
    return {'id':pid,'name':name,'mu':round(d.get('mu',0),2),'std':round(d.get('std',0),2),
            'matches': int(rows['match_id'].nunique()) if 'match_id' in rows.columns else 0}

def get_team_name(tid):
    rows = teams[teams['id'] == int(tid)]['name']
    return str(rows.iloc[0]) if not rows.empty else f'Team-{tid}'

def get_h2h(tidA, tidB):
    tnA = get_team_name(tidA); tnB = get_team_name(tidB)
    wA = wB = ties = 0
    recent = []
    for _, mrow in matches.iterrows():
        res = str(mrow.get('result',''))
        if tnA not in res or tnB not in res: continue
        if ' beat ' in res:
            winner = res.split(' beat ')[0].strip()
            if tnA in winner: wA += 1; recent.append({'winner': tnA, 'result': res})
            else:             wB += 1; recent.append({'winner': tnB, 'result': res})
        elif 'tied' in res.lower() or 'Tied' in res:
            ties += 1; recent.append({'winner': 'Tie', 'result': res})
    return {'wA':wA,'wB':wB,'ties':ties,'tnA':tnA,'tnB':tnB,'recent':recent[-5:]}

def get_squad_stats(tid):
    p = players[players['team_id'] == int(tid)].copy()
    if p.empty: return []
    agg = p.groupby('player_id').agg(
        avg_pts=('player_total_points','mean'),
        std_pts=('player_total_points','std'),
        matches=('match_id','nunique'),
        total_pts=('player_total_points','sum'),
        raid_pts=('player_raid_points_total','sum'),
        tackle_pts=('player_tackle_points_total','sum'),
    ).reset_index()
    names = players[['player_id','player_name']].drop_duplicates().set_index('player_id')['player_name']
    agg['name'] = agg['player_id'].map(names).fillna('-')
    agg = agg.sort_values('avg_pts', ascending=False)
    out = []
    for _, r in agg.iterrows():
        out.append({k: (round(float(r[k]),2) if pd.notna(r[k]) else 0)
                    for k in ['avg_pts','std_pts','total_pts','raid_pts','tackle_pts']})
        out[-1]['name']    = str(r['name'])
        out[-1]['matches'] = int(r['matches'])
    return out

def get_team_stats(tid):
    t_rows = teams[teams['id'] == int(tid)]
    avg_raid   = round(float(t_rows['raid_success'].mean())   * 100, 1) if not t_rows.empty else 0
    avg_tackle = round(float(t_rows['tackle_success'].mean()) * 100, 1) if not t_rows.empty else 0
    tn = get_team_name(tid)
    won = lost = tied = 0
    for _, mrow in matches.iterrows():
        res = str(mrow.get('result',''))
        if tn not in res: continue
        if ' beat ' in res:
            if res.strip().startswith(tn): won += 1
            else: lost += 1
        elif 'tied' in res.lower() or 'Tied' in res: tied += 1
    tr = t_rows.sort_values('match_id').tail(20)
    trend_x      = [f'M{i+1}' for i in range(len(tr))]
    raid_trend   = [round(float(x)*100,1) if pd.notna(x) else 0 for x in tr['raid_success'].tolist()]
    tackle_trend = [round(float(x)*100,1) if pd.notna(x) else 0 for x in tr['tackle_success'].tolist()]
    return {'avg_raid':avg_raid,'avg_tackle':avg_tackle,
            'wins':won,'losses':lost,'ties':tied,
            'trend_x':trend_x,'raid_trend':raid_trend,'tackle_trend':tackle_trend}

def save_match_to_db(rec):
    """Save to OUTPUT_DIR (avoids Windows permission errors on C:\\Users path)."""
    global players, matches, teams, player_dist, team_avg_mu, team_latest_stats
    new_mid = int(rec.get('match_id', int(matches['match_id'].max())+1 if not matches.empty else 9001))
    perf    = rec.get('player_performances', [])

    # 1. Append new player rows (dedup by player_id + match_id)
    if perf:
        new_p = pd.DataFrame([{
            'player_id'               : int(p['pid']),
            'player_name'             : str(p.get('name','')),
            'player_total_points'     : float(p.get('points',0)),
            'player_raid_points_total': float(p.get('raid_pts',0)),
            'player_tackle_points_total': float(p.get('tackle_pts',0)),
            'match_id': new_mid,
            'team_id' : int(p.get('team_id',0)),
        } for p in perf])
        # Drop any duplicate (player_id, match_id) rows already in players
        existing_keys = set(zip(players['player_id'].astype(int), players['match_id'].astype(int)))
        new_p = new_p[~new_p.apply(lambda r: (int(r['player_id']),int(r['match_id'])) in existing_keys, axis=1)]
        players = pd.concat([players, new_p], ignore_index=True)

    # 2. Append match row only if match_id not already present
    if new_mid not in matches['match_id'].astype(int).values:
        new_m = pd.DataFrame([{
            'match_id':new_mid,'result':rec.get('result',''),
            'date':rec.get('date',''),'series_name':'Live Match','status':'Full Time',
        }])
        matches = pd.concat([matches, new_m], ignore_index=True)

    # 3. New team rows
    team_perf = {}
    for p in perf:
        team_perf.setdefault(int(p.get('team_id',0)), []).append(float(p.get('points',0)))

    if team_perf and match_state.get('features'):
        feats = match_state['features']
        new_t = []
        for tid, pts_list in team_perf.items():
            is_A = (tid == match_state.get('teamA_id'))
            rs   = feats.get('raid_success_A'   if is_A else 'raid_success_B',   0.5)
            ts   = feats.get('tackle_success_A' if is_A else 'tackle_success_B', 0.4)
            scr  = match_state.get('scoreA' if is_A else 'scoreB', 0)
            new_t.append({'id':tid,'match_id':new_mid,'score':scr,
                'stats.raids.successful':round(rs*10),'stats.raids.total':10,
                'stats.tackles.successful':round(ts*10),'stats.tackles.total':10})
        # Only append team rows for (id, match_id) combos not already present
        existing_tm = set(zip(teams['id'].astype(int), teams['match_id'].astype(int))) if 'match_id' in teams.columns else set()
        new_t_filtered = [r for r in new_t if (int(r['id']),int(r['match_id'])) not in existing_tm]
        if new_t_filtered:
            teams = pd.concat([teams, pd.DataFrame(new_t_filtered)], ignore_index=True)
        # Recompute raid/tackle_success columns
        teams['raid_success']   = np.where(teams['stats.raids.total']>0,
            teams['stats.raids.successful']/teams['stats.raids.total'], 0.0)
        teams['tackle_success'] = np.where(teams['stats.tackles.total']>0,
            teams['stats.tackles.successful']/teams['stats.tackles.total'], 0.0)
        for row in new_t:
            tid2 = row['id']
            team_latest_stats[tid2] = {
                'raid_success'  : row['stats.raids.successful']/10,
                'tackle_success': row['stats.tackles.successful']/10,
            }

    # 4. Refresh player_dist
    for p in perf:
        pid = int(p['pid'])
        pts_col = players[players['player_id']==pid]['player_total_points'].dropna()
        if len(pts_col) > 2:
            player_dist[pid] = {'mu':float(pts_col.mean()), 'std':float(max(pts_col.std(),0.1))}

    # 5. Refresh team_avg_mu
    for tid in team_perf:
        pids = players[players['team_id']==tid]['player_id'].unique()
        mus  = [player_dist.get(int(px),{}).get('mu',0) for px in pids]
        team_avg_mu[tid] = float(np.mean(mus)) if mus else 1.0

    # 6. Write CSVs to OUTPUT_DIR (writable path)
    saved = []; errors = []
    for fname, df in [('PKL_players.csv',players),('PKL_matches.csv',matches),('PKL_teams.csv',teams)]:
        path = os.path.join(OUTPUT_DIR, fname)
        try:
            df.to_csv(path, index=False)
            saved.append(fname)
        except Exception as e:
            errors.append(f'{fname}: {e}')

    if errors:
        return False, f'In-memory updated. Disk errors: {"; ".join(errors)}'
    return True, f'Saved {len(saved)} files to {OUTPUT_DIR}'

# ------------------------------------------------------------------ Flask app
app = Flask(__name__)
match_state = dict(teamA_id=None, teamB_id=None,
    starters_A=[], bench_A=[], starters_B=[], bench_B=[],
    scoreA=0, scoreB=0, features={}, probA=0.5)

HTML_PAGE = (
    "<!DOCTYPE html>\n"
    "<html lang='en'>\n"
    "<head>\n"
    "<meta charset='UTF-8'>\n"
    "<meta name='viewport' content='width=device-width,initial-scale=1'>\n"
    "<title>Kabaddi Coach Dashboard</title>\n"
    "<script src='https://cdn.jsdelivr.net/npm/chart.js@4.4.0/dist/chart.umd.min.js'></script>\n"
    "<style>\n"
    ":root{--bg:#0d1117;--card:#161b22;--border:#30363d;--text:#e6edf3;--muted:#7d8590;"
    "--green:#3fb950;--red:#f85149;--blue:#58a6ff;--yellow:#d29922;}\n"
    "*{box-sizing:border-box;margin:0;padding:0}\n"
    "body{background:var(--bg);color:var(--text);font-family:'Segoe UI',system-ui,sans-serif;min-height:100vh}\n"
    "header{background:linear-gradient(135deg,#1a2332,#0d1117);border-bottom:1px solid var(--border);"
    "padding:14px 28px;display:flex;align-items:center;gap:14px;position:sticky;top:0;z-index:100}\n"
    ".h1{font-size:1.35rem;font-weight:700;background:linear-gradient(90deg,#58a6ff,#3fb950);"
    "-webkit-background-clip:text;-webkit-text-fill-color:transparent}\n"
    ".sub{color:var(--muted);font-size:.82rem}\n"
    ".tab-bar{display:flex;gap:2px;padding:14px 28px 0;border-bottom:1px solid var(--border);background:var(--card)}\n"
    ".tab{padding:9px 18px;cursor:pointer;border-radius:8px 8px 0 0;color:var(--muted);"
    "font-size:.88rem;border:none;background:none;transition:.2s}\n"
    ".tab.active{background:var(--bg);color:var(--text);border:1px solid var(--border);"
    "border-bottom:1px solid var(--bg);margin-bottom:-1px}\n"
    ".tab:hover:not(.active){color:var(--text)}\n"
    ".panel{display:none;padding:22px 28px;max-width:1400px;margin:0 auto;animation:fi .2s}\n"
    ".panel.active{display:block}\n"
    "@keyframes fi{from{opacity:0;transform:translateY(4px)}to{opacity:1;transform:none}}\n"
    ".g2{display:grid;grid-template-columns:1fr 1fr;gap:14px}\n"
    ".g3{display:grid;grid-template-columns:1fr 1fr 1fr;gap:14px}\n"
    ".card{background:var(--card);border:1px solid var(--border);border-radius:12px;padding:18px}\n"
    ".card h3{font-size:.88rem;color:var(--muted);font-weight:600;text-transform:uppercase;"
    "letter-spacing:.05em;margin-bottom:14px}\n"
    "select,input[type=number]{background:#0d1117;color:var(--text);border:1px solid var(--border);"
    "border-radius:8px;padding:9px 13px;font-size:.88rem;width:100%;outline:none;transition:.2s}\n"
    "select:focus,input:focus{border-color:var(--blue)}\n"
    ".btn{padding:9px 18px;border-radius:8px;border:none;font-size:.88rem;font-weight:600;"
    "cursor:pointer;transition:.2s;display:inline-flex;align-items:center;gap:5px}\n"
    ".bp{background:var(--blue);color:#000}.bp:hover{background:#79c0ff}\n"
    ".bs{background:var(--green);color:#000}.bs:hover{background:#56d364}\n"
    ".bg-btn{background:transparent;color:var(--text);border:1px solid var(--border)}.bg-btn:hover{background:var(--border)}\n"
    ".sm{padding:5px 12px;font-size:.8rem}\n"
    ".pw{border-radius:8px;overflow:hidden;height:38px;display:flex;margin:10px 0}\n"
    ".pa{background:linear-gradient(90deg,#1f6feb,#388bfd);display:flex;align-items:center;"
    "justify-content:center;color:#fff;font-weight:700;transition:width .5s;font-size:.9rem;min-width:55px}\n"
    ".pb{background:linear-gradient(90deg,#da3633,#f85149);display:flex;align-items:center;"
    "justify-content:center;color:#fff;font-weight:700;transition:width .5s;font-size:.9rem;min-width:55px}\n"
    ".pgrid{display:grid;grid-template-columns:repeat(auto-fill,minmax(155px,1fr));gap:7px;"
    "max-height:400px;overflow-y:auto;padding-right:3px}\n"
    ".pc{background:#0d1117;border:2px solid var(--border);border-radius:9px;padding:10px;"
    "cursor:pointer;transition:.2s;text-align:center}\n"
    ".pc.starter{border-color:var(--green);background:rgba(63,185,80,.08)}\n"
    ".pc.bench{border-color:var(--yellow);background:rgba(210,153,34,.08)}\n"
    ".pc:hover{border-color:var(--blue);transform:translateY(-2px)}\n"
    ".pn{font-size:.78rem;font-weight:600;margin-bottom:3px;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}\n"
    ".pp{font-size:.7rem;color:var(--muted)}\n"
    ".pr{font-size:.67rem;margin-top:3px;padding:2px 7px;border-radius:99px;display:inline-block}\n"
    ".rs{background:rgba(63,185,80,.2);color:var(--green)}\n"
    ".rb{background:rgba(210,153,34,.2);color:var(--yellow)}\n"
    ".badge{display:inline-block;padding:3px 9px;border-radius:99px;font-size:.72rem;font-weight:600}\n"
    ".bb{background:rgba(88,166,255,.15);color:var(--blue)}\n"
    ".bg2{background:rgba(63,185,80,.15);color:var(--green)}\n"
    ".by{background:rgba(210,153,34,.15);color:var(--yellow)}\n"
    ".br{background:rgba(248,81,73,.15);color:var(--red)}\n"
    ".sr{display:flex;justify-content:space-between;padding:7px 0;border-bottom:1px solid var(--border)}\n"
    ".sr:last-child{border-bottom:none}\n"
    ".sl{color:var(--muted);font-size:.83rem}\n"
    ".sv{font-weight:700;font-size:.88rem}\n"
    ".sdis{display:flex;align-items:center;justify-content:center;gap:18px;padding:14px}\n"
    ".st{text-align:center}.stn{font-size:.82rem;color:var(--muted)}\n"
    ".ssc{font-size:2.4rem;font-weight:800;line-height:1}\n"
    ".step-bar{display:flex;margin-bottom:20px}\n"
    ".step{flex:1;text-align:center;padding:9px 4px;font-size:.78rem;color:var(--muted);"
    "border-bottom:2px solid var(--border);transition:.2s}\n"
    ".step.done{color:var(--green);border-color:var(--green)}\n"
    ".step.active{color:var(--blue);border-color:var(--blue);font-weight:700}\n"
    ".alert{padding:10px 14px;border-radius:8px;font-size:.85rem;margin:7px 0}\n"
    ".ai{background:rgba(88,166,255,.1);border:1px solid rgba(88,166,255,.3);color:var(--blue)}\n"
    ".as{background:rgba(63,185,80,.1);border:1px solid rgba(63,185,80,.3);color:var(--green)}\n"
    ".aw{background:rgba(210,153,34,.1);border:1px solid rgba(210,153,34,.3);color:var(--yellow)}\n"
    ".ad{background:rgba(248,81,73,.1);border:1px solid rgba(248,81,73,.3);color:var(--red)}\n"
    ".why{background:#0d1117;border-left:3px solid var(--blue);padding:10px 14px;"
    "border-radius:0 7px 7px 0;font-size:.8rem;line-height:1.6;color:#b1bac4;margin-top:10px}\n"
    ".fg{margin-bottom:12px}.fgl{display:block;font-size:.82rem;color:var(--muted);margin-bottom:5px}\n"
    "table{width:100%;border-collapse:collapse;font-size:.83rem}\n"
    "th{text-align:left;padding:9px 11px;color:var(--muted);font-weight:600;"
    "border-bottom:1px solid var(--border);white-space:nowrap}\n"
    "td{padding:8px 11px;border-bottom:1px solid #21262d}\n"
    "tr:hover td{background:rgba(255,255,255,.03)}\n"
    "canvas{max-height:220px}\n"
    "::-webkit-scrollbar{width:5px;height:5px}\n"
    "::-webkit-scrollbar-thumb{background:var(--border);border-radius:3px}\n"
    ".ld{display:inline-block;width:14px;height:14px;border:2px solid var(--border);"
    "border-top-color:var(--blue);border-radius:50%;animation:sp .6s linear infinite}\n"
    "@keyframes sp{to{transform:rotate(360deg)}}\n"
    ".mt8{margin-top:8px}.mt14{margin-top:14px}.mt20{margin-top:20px}\n"
    ".mb8{margin-bottom:8px}.mb14{margin-bottom:14px}\n"
    ".fl{display:flex}.gap8{gap:8px}.aic{align-items:center}.jb{justify-content:space-between}\n"
    ".wf{width:100%}.mla{margin-left:auto}\n"
    "</style>\n"
    "</head>\n"
    "<body>\n"
    "<header>\n"
    "  <div style='font-size:1.9rem'>&#127942;</div>\n"
    "  <div><div class='h1'>Kabaddi Coach Intelligence Dashboard</div>\n"
    "    <div class='sub'>Pro Kabaddi League &middot; 5-Model ML &middot; Bayesian Live Updates &middot; Save to CSV</div>\n"
    "  </div>\n"
    "  <div class='mla fl gap8'>\n"
    "    <button class='btn bg-btn sm' onclick='resetMatch()'>&#8634; New Match</button>\n"
    "  </div>\n"
    "</header>\n"
    "<div class='tab-bar'>\n"
    "  <button class='tab active' onclick=\"showTab('setup')\"    id='tab-setup'>1. Setup</button>\n"
    "  <button class='tab'        onclick=\"showTab('lineup')\"   id='tab-lineup'>2. Lineup</button>\n"
    "  <button class='tab'        onclick=\"showTab('match')\"    id='tab-match'>3. Live Match</button>\n"
    "  <button class='tab'        onclick=\"showTab('report')\"   id='tab-report'>H2H Report</button>\n"
    "</div>\n"
    "\n"
    "<!-- SETUP -->\n"
    "<div class='panel active' id='panel-setup'>\n"
    "  <div class='step-bar'>\n"
    "    <div class='step active' id='step1'>1 - Select Teams</div>\n"
    "    <div class='step' id='step2'>2 - Lineup</div>\n"
    "    <div class='step' id='step3'>3 - Live Match</div>\n"
    "    <div class='step' id='step4'>4 - Done</div>\n"
    "  </div>\n"
    "  <div class='g2' style='max-width:680px'>\n"
    "    <div class='card'><h3>Team A (Green)</h3>\n"
    "      <select id='dd_A'><option value=''>Select Team A...</option></select>\n"
    "      <div id='prevA' class='mt14'></div></div>\n"
    "    <div class='card'><h3>Team B (Red)</h3>\n"
    "      <select id='dd_B'><option value=''>Select Team B...</option></select>\n"
    "      <div id='prevB' class='mt14'></div></div>\n"
    "  </div>\n"
    "  <div class='mt14'>\n"
    "    <button class='btn bp' onclick='confirmTeams()'>Confirm Teams - Go to Lineup</button>\n"
    "  </div>\n"
    "  <div id='setup-msg' class='mt8'></div>\n"
    "  <div class='card mt20' style='max-width:680px'>\n"
    "    <h3>How Win Probability Works</h3>\n"
    "    <div class='why'>\n"
    "      <b>5 ML models compete:</b> Logistic Regression, Random Forest, Gradient Boosting,\n"
    "      Gaussian Naive Bayes, SVM (calibrated). Best cross-validated ROC-AUC wins the election.\n"
    "      <br><br>\n"
    "      <b>Bayesian live update:</b> When a player scores far above/below their career average\n"
    "      (mu +/- sigma), Bayes theorem shifts probability without retraining.\n"
    "      Formula: posterior odds = prior odds x exp(alpha x z-score). Alpha=0.4.\n"
    "      <br><br>\n"
    "      <b>Score-diff augmentation:</b> Real matches all start 0-0. We synthetically create\n"
    "      training copies at score_diff in {-20, -12, -6, -2, 0, +2, +6, +12, +20} so the\n"
    "      model learns that score difference matters. Without this the score slider would have\n"
    "      zero effect on probability.\n"
    "      <br><br>\n"
    "      <b>Realistic ROC-AUC:</b> 0.65-0.75 for PKL data. Values above 0.90 indicate\n"
    "      overfitting, not genuine accuracy.\n"
    "    </div>\n"
    "  </div>\n"
    "</div>\n"
    "\n"
    "<!-- LINEUP -->\n"
    "<div class='panel' id='panel-lineup'>\n"
    "  <div class='fl jb aic mb14'>\n"
    "    <div><div style='font-size:1rem;font-weight:700' id='lineup-title'>Pick Lineup</div>\n"
    "      <div style='font-size:.8rem;color:var(--muted)'>Click to toggle Starter (green) / Bench (yellow). Max 7 starters.</div></div>\n"
    "    <button class='btn bs' onclick='confirmLineup()'>Confirm Lineup</button>\n"
    "  </div>\n"
    "  <div class='g2'>\n"
    "    <div class='card'><h3 id='la-title'>Team A</h3>\n"
    "      <div class='fl gap8 mb8'>\n"
    "        <span class='badge bg2' id='sc-A'>0 starters</span>\n"
    "        <span class='badge by' id='bc-A'>0 bench</span></div>\n"
    "      <div class='pgrid' id='pgrid-A'></div></div>\n"
    "    <div class='card'><h3 id='lb-title'>Team B</h3>\n"
    "      <div class='fl gap8 mb8'>\n"
    "        <span class='badge bg2' id='sc-B'>0 starters</span>\n"
    "        <span class='badge by' id='bc-B'>0 bench</span></div>\n"
    "      <div class='pgrid' id='pgrid-B'></div></div>\n"
    "  </div>\n"
    "  <div id='lineup-msg' class='mt8'></div>\n"
    "</div>\n"
    "\n"
    "<!-- LIVE MATCH -->\n"
    "<div class='panel' id='panel-match'>\n"
    "  <div class='card mb14'>\n"
    "    <div class='sdis'>\n"
    "      <div class='st'><div class='stn' id='mn-A'>Team A</div>\n"
    "        <div class='ssc' style='color:var(--blue)' id='ms-A'>0</div></div>\n"
    "      <div style='font-size:1.3rem;color:var(--muted)'>VS</div>\n"
    "      <div class='st'><div class='stn' id='mn-B'>Team B</div>\n"
    "        <div class='ssc' style='color:var(--red)' id='ms-B'>0</div></div>\n"
    "    </div>\n"
    "    <div class='pw'><div class='pa' id='pbar-a' style='width:50%'>50%</div>\n"
    "      <div class='pb' id='pbar-b' style='width:50%'>50%</div></div>\n"
    "    <div id='feat-row' style='font-size:.77rem;color:var(--muted);text-align:center;margin-top:6px'></div>\n"
    "  </div>\n"
    "  <div class='g3'>\n"
    "    <div class='card'>\n"
    "      <h3>Update Score</h3>\n"
    "      <div class='fg'><label class='fgl' id='sl-A'>Team A Score</label>\n"
    "        <input type='number' id='inp-A' value='0' min='0' max='80'></div>\n"
    "      <div class='fg'><label class='fgl' id='sl-B'>Team B Score</label>\n"
    "        <input type='number' id='inp-B' value='0' min='0' max='80'></div>\n"
    "      <button class='btn bp wf' onclick='updateScore()'>Update and Predict</button>\n"
    "      <div class='why mt8' style='font-size:.77rem'>\n"
    "        Updates score_diff feature. Model recomputes probability instantly. Can repeat as many times as needed.\n"
    "      </div>\n"
    "    </div>\n"
    "    <div class='card'>\n"
    "      <h3>Player Performance - Bayesian Update</h3>\n"
    "      <div class='fg'><label class='fgl'>Player on court</label>\n"
    "        <select id='bay-pid'></select></div>\n"
    "      <div class='fg'>\n"
    "        <label class='fgl'>Expected: <span id='bay-exp' style='color:var(--blue)'>-</span></label>\n"
    "        <label class='fgl'>Actual points scored</label>\n"
    "        <input type='number' id='bay-obs' value='0' min='0' max='40'></div>\n"
    "      <button class='btn bp wf' onclick='applyBayes()'>Apply Bayesian Update</button>\n"
    "      <div id='bay-res' class='mt8'></div>\n"
    "      <div class='why mt8' style='font-size:.77rem'>\n"
    "        LR = exp(0.4 x (actual-mu)/sigma). Above average = LR > 1 = prob shifts toward that team.\n"
    "      </div>\n"
    "    </div>\n"
    "    <div class='card'>\n"
    "      <h3>Substitution</h3>\n"
    "      <div class='fg'><label class='fgl'>Team</label>\n"
    "        <select id='sub-side' onchange='refreshSubs()'>\n"
    "          <option value='A'>Team A</option><option value='B'>Team B</option></select></div>\n"
    "      <div class='fg'><label class='fgl'>Remove from starters</label>\n"
    "        <select id='sub-out'></select></div>\n"
    "      <div class='fg'><label class='fgl'>Bring in from bench only</label>\n"
    "        <select id='sub-in'></select></div>\n"
    "      <div class='fl gap8'>\n"
    "        <button class='btn bg-btn' style='flex:1' onclick='previewSub()'>Preview</button>\n"
    "        <button class='btn bs' style='flex:1' onclick='applySub()'>Apply Sub</button></div>\n"
    "      <div id='sub-res' class='mt8'></div>\n"
    "    </div>\n"
    "  </div>\n"
    "  <div class='card mt14'><h3>Win Probability Timeline</h3>\n"
    "    <canvas id='chart-prob' style='max-height:200px'></canvas></div>\n"
    "  <div class='card mt14'>\n"
    "    <h3>Save Match to Database (PKL_players.csv + PKL_matches.csv + PKL_teams.csv)</h3>\n"
    "    <div style='font-size:.82rem;color:var(--muted);margin-bottom:12px'>\n"
    "      Enter actual points per player. Click Save to write all 3 CSV files to your Output folder.\n"
    "    </div>\n"
    "    <div class='g2' id='perf-grid'></div>\n"
    "    <div class='fl gap8 mt14'>\n"
    "      <button class='btn bs' onclick='saveMatch()'>Save All 3 CSVs</button></div>\n"
    "    <div id='save-msg' class='mt8'></div>\n"
    "  </div>\n"
    "</div>\n"
    "\n"
   
    "<!-- H2H REPORT -->\n"
    "<div class='panel' id='panel-report'>\n"
    "  <div class='card' style='max-width:540px;margin-bottom:16px'>\n"
    "    <h3>Head-to-Head Report</h3>\n"
    "    <div class='fg'><label class='fgl'>Team A</label>\n"
    "      <select id='h2h-A'><option value=''>Select...</option></select></div>\n"
    "    <div class='fg'><label class='fgl'>Team B</label>\n"
    "      <select id='h2h-B'><option value=''>Select...</option></select></div>\n"
    "    <button class='btn bp mt8' onclick='loadH2H()'>Generate H2H Report</button>\n"
    "  </div>\n"
    "  <div id='h2h-content'><div class='alert ai'>Select two teams and click Generate.</div></div>\n"
    "</div>\n"
    "\n"
    "<script>\n"
    "let TM={},RA={starters:[],bench:[],all:[]},RB={starters:[],bench:[],all:[]};\n"
    "let AP=[],PR={},probHist=[],probChart=null;\n"
    "\n"
    "async function init(){\n"
    "  const d=await (await fetch('/api/teams')).json(); TM=d;\n"
    "  ['dd_A','dd_B','h2h-A','h2h-B'].forEach(sid=>{\n"
    "    const s=document.getElementById(sid);\n"
    "    if(!s) return;\n"
    "    Object.entries(d).forEach(([id,name])=>s.appendChild(new Option(name,id)));\n"
    "  });\n"
    "  document.getElementById('dd_A').addEventListener('change',()=>prevTeam('A'));\n"
    "  document.getElementById('dd_B').addEventListener('change',()=>prevTeam('B'));\n"
    "  document.getElementById('bay-pid').addEventListener('change',updBayExp);\n"
    "}\n"
    "\n"
    "function showTab(t){\n"
    "  document.querySelectorAll('.panel').forEach(p=>p.classList.remove('active'));\n"
    "  document.querySelectorAll('.tab').forEach(b=>b.classList.remove('active'));\n"
    "  document.getElementById('panel-'+t).classList.add('active');\n"
    "  document.getElementById('tab-'+t).classList.add('active');\n"
    "}\n"
    "\n"
    "async function prevTeam(side){\n"
    "  const tid=document.getElementById('dd_'+side).value; if(!tid) return;\n"
    "  const d=await (await fetch('/api/team_stats?tid='+tid)).json();\n"
    "  document.getElementById('prev'+side).innerHTML=\n"
    "    '<div class=\"sr\"><span class=\"sl\">Raid Success</span><span class=\"sv\">'+d.avg_raid+'%</span></div>'+\n"
    "    '<div class=\"sr\"><span class=\"sl\">Tackle Success</span><span class=\"sv\">'+d.avg_tackle+'%</span></div>'+\n"
    "    '<div class=\"sr\"><span class=\"sl\">W / L / T</span><span class=\"sv\">'+d.wins+' / '+d.losses+' / '+d.ties+'</span></div>';\n"
    "}\n"
    "\n"
    "async function confirmTeams(){\n"
    "  const tA=document.getElementById('dd_A').value,tB=document.getElementById('dd_B').value;\n"
    "  const msg=document.getElementById('setup-msg');\n"
    "  if(!tA||!tB){msg.innerHTML='<div class=\"alert aw\">Select both teams.</div>';return;}\n"
    "  if(tA===tB){msg.innerHTML='<div class=\"alert ad\">Choose two DIFFERENT teams.</div>';return;}\n"
    "  msg.innerHTML='<div class=\"alert ai\"><span class=\"ld\"></span> Loading rosters...</div>';\n"
    "  const d=await (await fetch('/api/roster?tidA='+tA+'&tidB='+tB)).json();\n"
    "  RA=d.rA; RB=d.rB; AP=d.all;\n"
    "  buildGrid('A',RA,tA); buildGrid('B',RB,tB);\n"
    "  document.getElementById('la-title').textContent='Team A: '+TM[tA];\n"
    "  document.getElementById('lb-title').textContent='Team B: '+TM[tB];\n"
    "  document.getElementById('lineup-title').textContent=TM[tA]+' vs '+TM[tB]+' - Pick Lineup';\n"
    "  document.getElementById('step1').className='step done';\n"
    "  document.getElementById('step2').className='step active';\n"
    "  msg.innerHTML='<div class=\"alert as\">Teams confirmed. Go to Lineup tab.</div>';\n"
    "  showTab('lineup');\n"
    "}\n"
    "\n"
    "function buildGrid(side,roster,tid){\n"
    "  const c=document.getElementById('pgrid-'+side); c.innerHTML='';\n"
    "  roster.starters.forEach(p=>PR[side+Number(p.id)]='starter');\n"
    "  roster.bench.forEach(p=>PR[side+Number(p.id)]='bench');\n"
    "  roster.all.forEach(p=>{\n"
    "    const role=PR[side+p.id]||'bench';\n"
    "    const el=document.createElement('div');\n"
    "    el.className='pc '+role; el.id='pc-'+side+'-'+Number(p.id);\n"
    "    el.innerHTML='<div class=\"pn\" title=\"'+p.name+'\">'+p.name+'</div>'\n"
    "      +'<div class=\"pp\">avg '+p.mu+'pts - '+p.matches+'m</div>'\n"
    "      +'<div class=\"pr r'+(role==='starter'?'s':'b')+'\" id=\"pr-'+side+'-'+p.id+'\">'+role.toUpperCase()+'</div>';\n"
    "    el.onclick=()=>toggleP(side,Number(p.id));\n"
    "    c.appendChild(el);\n"
    "  });\n"
    "  updCounts(side);\n"
    "}\n"
    "\n"
    "function toggleP(side,pid){\n"
    "  pid=Number(pid); const key=side+pid,cur=PR[key]||'bench';\n"
    "  if(cur==='starter') PR[key]='bench';\n"
    "  else{ if(getPids(side,'starter').length>=7){alert('Max 7 starters.');return;} PR[key]='starter'; }\n"
    "  const nr=PR[key];\n"
    "  const card=document.getElementById('pc-'+side+'-'+pid);\n"
    "  const badge=document.getElementById('pr-'+side+'-'+pid);\n"
    "  if(card) card.className='pc '+nr;\n"
    "  if(badge){badge.className='pr r'+(nr==='starter'?'s':'b');badge.textContent=nr.toUpperCase();}\n"
    "  updCounts(side);\n"
    "}\n"
    "\n"
    "function getPids(side,role){\n"
    "  const r=side==='A'?RA:RB;\n"
    "  return r.all.map(p=>Number(p.id)).filter(id=>PR[side+id]===role);\n"
    "}\n"
    "\n"
    "function updCounts(side){\n"
    "  document.getElementById('sc-'+side).textContent=getPids(side,'starter').length+' starters';\n"
    "  document.getElementById('bc-'+side).textContent=getPids(side,'bench').length+' bench';\n"
    "}\n"
    "\n"
    "async function confirmLineup(){\n"
    "  const stA=getPids('A','starter'),bnA=getPids('A','bench');\n"
    "  const stB=getPids('B','starter'),bnB=getPids('B','bench');\n"
    "  const msg=document.getElementById('lineup-msg');\n"
    "  if(stA.length<5||stB.length<5){msg.innerHTML='<div class=\"alert aw\">Need 5+ starters per team.</div>';return;}\n"
    "  const tA=document.getElementById('dd_A').value,tB=document.getElementById('dd_B').value;\n"
    "  msg.innerHTML='<div class=\"alert ai\"><span class=\"ld\"></span> Computing...</div>';\n"
    "  const d=await (await fetch('/api/confirm_lineup',{\n"
    "    method:'POST',headers:{'Content-Type':'application/json'},\n"
    "    body:JSON.stringify({tA,tB,stA,bnA,stB,bnB})\n"
    "  })).json();\n"
    "  if(d.error){msg.innerHTML='<div class=\"alert ad\">'+d.error+'</div>';return;}\n"
    "  msg.innerHTML='<div class=\"alert as\">Lineup confirmed. Going to Live Match tab.</div>';\n"
    "  document.getElementById('step2').className='step done';\n"
    "  document.getElementById('step3').className='step active';\n"
    "  initMatch(d); showTab('match');\n"
    "}\n"
    "\n"
    "function initMatch(d){\n"
    "  const tA=document.getElementById('dd_A').value,tB=document.getElementById('dd_B').value;\n"
    "  ['mn-A','sl-A'].forEach(id=>document.getElementById(id).textContent=TM[tA]);\n"
    "  ['mn-B','sl-B'].forEach(id=>document.getElementById(id).textContent=TM[tB]);\n"
    "  probHist=[d.probA]; updProb(d.probA,d.features);\n"
    "  buildProbChart(); buildBayDd(); refreshSubs(); buildPerfGrid();\n"
    "}\n"
    "\n"
    "function updProb(pA,feats){\n"
    "  const pct=Math.round(pA*100);\n"
    "  const tA=document.getElementById('dd_A').value,tB=document.getElementById('dd_B').value;\n"
    "  const ba=document.getElementById('pbar-a'),bb=document.getElementById('pbar-b');\n"
    "  ba.style.width=pct+'%'; ba.textContent=TM[tA]+' '+pct+'%';\n"
    "  bb.style.width=(100-pct)+'%'; bb.textContent=TM[tB]+' '+(100-pct)+'%';\n"
    "  if(feats) document.getElementById('feat-row').textContent=\n"
    "    'RaidA:'+(feats.raid_success_A*100).toFixed(0)+'% TkA:'+(feats.tackle_success_A*100).toFixed(0)+'% | '\n"
    "    +'RaidB:'+(feats.raid_success_B*100).toFixed(0)+'% TkB:'+(feats.tackle_success_B*100).toFixed(0)+'% | '\n"
    "    +'Diff:'+(feats.score_diff>=0?'+':'')+feats.score_diff;\n"
    "}\n"
    "\n"
    "function buildProbChart(){\n"
    "  if(probChart) probChart.destroy();\n"
    "  const ctx=document.getElementById('chart-prob').getContext('2d');\n"
    "  probChart=new Chart(ctx,{type:'line',\n"
    "    data:{labels:['Start'],\n"
    "      datasets:[{label:'Team A Win %',data:[Math.round(probHist[0]*100)],\n"
    "        borderColor:'#58a6ff',backgroundColor:'rgba(88,166,255,.15)',\n"
    "        fill:true,tension:.3,pointRadius:4}]},\n"
    "    options:{responsive:true,\n"
    "      scales:{y:{min:0,max:100,ticks:{color:'#7d8590'},grid:{color:'#21262d'}},\n"
    "              x:{ticks:{color:'#7d8590'},grid:{color:'#21262d'}}},\n"
    "      plugins:{legend:{labels:{color:'#e6edf3'}}}}});\n"
    "}\n"
    "\n"
    "function pushProb(pA,lbl){\n"
    "  probHist.push(pA);\n"
    "  if(probChart){probChart.data.labels.push(lbl||('Ev'+probHist.length));\n"
    "    probChart.data.datasets[0].data.push(Math.round(pA*100)); probChart.update();}\n"
    "}\n"
    "\n"
    "async function updateScore(){\n"
    "  const sA=parseInt(document.getElementById('inp-A').value)||0;\n"
    "  const sB=parseInt(document.getElementById('inp-B').value)||0;\n"
    "  const d=await (await fetch('/api/update_score',{\n"
    "    method:'POST',headers:{'Content-Type':'application/json'},\n"
    "    body:JSON.stringify({scoreA:sA,scoreB:sB})\n"
    "  })).json();\n"
    "  document.getElementById('ms-A').textContent=sA;\n"
    "  document.getElementById('ms-B').textContent=sB;\n"
    "  updProb(d.probA,d.features); pushProb(d.probA,sA+'-'+sB);\n"
    "}\n"
    "\n"
    "function buildBayDd(){\n"
    "  const sel=document.getElementById('bay-pid'); sel.innerHTML='';\n"
    "  const tA=document.getElementById('dd_A').value,tB=document.getElementById('dd_B').value;\n"
    "  const stA=getPids('A','starter'),stB=getPids('B','starter');\n"
    "  [...stA,...stB].forEach(pid=>{\n"
    "    const p=AP.find(x=>x.id===pid)||{name:'Player '+pid,mu:0};\n"
    "    const side=stA.includes(pid)?TM[tA]:TM[tB];\n"
    "    sel.appendChild(new Option(p.name+' ('+side+', avg '+p.mu+'pts)',pid));\n"
    "  });\n"
    "  updBayExp();\n"
    "}\n"
    "\n"
    "function updBayExp(){\n"
    "  const pid=parseInt(document.getElementById('bay-pid').value);\n"
    "  const p=AP.find(x=>x.id===pid)||{};\n"
    "  document.getElementById('bay-exp').textContent=\n"
    "    p.mu!==undefined?p.mu+' pts (+/-'+p.std+')':'-';\n"
    "}\n"
    "\n"
    "async function applyBayes(){\n"
    "  const pid=parseInt(document.getElementById('bay-pid').value);\n"
    "  const obs=parseFloat(document.getElementById('bay-obs').value)||0;\n"
    "  const p=AP.find(x=>x.id===pid)||{name:'Player',mu:0};\n"
    "  const d=await (await fetch('/api/bayes_update',{\n"
    "    method:'POST',headers:{'Content-Type':'application/json'},\n"
    "    body:JSON.stringify({pid,observed:obs})\n"
    "  })).json();\n"
    "  const diff=Math.round((d.probA-d.prior)*100),up=diff>=0;\n"
    "  const col=up?'var(--green)':'var(--red)';\n"
    "  document.getElementById('bay-res').innerHTML=\n"
    "    '<div class=\"alert\" style=\"border-color:'+col+';color:'+col+';background:rgba(0,0,0,.2)\">'\n"
    "    +p.name+': actual '+obs+' vs expected '+p.mu+'pts<br>'\n"
    "    +'Prob: '+Math.round(d.prior*100)+'% to <strong>'+Math.round(d.probA*100)+'%</strong>'\n"
    "    +' ('+(up?'up +':'down ')+Math.abs(diff)+'%)'\n"
    "    +'</div>';\n"
    "  updProb(d.probA,null); pushProb(d.probA,'Bayes:'+p.name.split(' ')[0]);\n"
    "}\n"
    "\n"
    "function refreshSubs(){\n"
    "  const side=document.getElementById('sub-side').value;\n"
    "  const selOut=document.getElementById('sub-out'),selIn=document.getElementById('sub-in');\n"
    "  selOut.innerHTML=''; selIn.innerHTML='';\n"
    "  const starters=getPids(side,'starter'),bench=getPids(side,'bench');\n"
    "  const R=side==='A'?RA:RB;\n"
    "  if(!starters.length){selOut.innerHTML='<option>No starters set</option>';}\n"
    "  else starters.forEach(pid=>{\n"
    "    const p=R.all.find(x=>Number(x.id)===Number(pid))||{name:'Player '+pid,mu:0};\n"
    "    selOut.appendChild(new Option(p.name+' (avg '+p.mu+'pts)',Number(pid)));\n"
    "  });\n"
    "  if(!bench.length){selIn.innerHTML='<option>No bench players</option>';}\n"
    "  else bench.forEach(pid=>{\n"
    "    const p=R.all.find(x=>Number(x.id)===Number(pid))||{name:'Player '+pid,mu:0};\n"
    "    selIn.appendChild(new Option(p.name+' (avg '+p.mu+'pts)',Number(pid)));\n"
    "  });\n"
    "}\n"
    "\n"
    "async function previewSub(){\n"
    "  const side=document.getElementById('sub-side').value;\n"
    "  const outP=parseInt(document.getElementById('sub-out').value);\n"
    "  const inP=parseInt(document.getElementById('sub-in').value);\n"
    "  if(!outP||!inP){document.getElementById('sub-res').innerHTML='<div class=\"alert aw\">Select players.</div>';return;}\n"
    "  const d=await (await fetch('/api/simulate_sub',{\n"
    "    method:'POST',headers:{'Content-Type':'application/json'},\n"
    "    body:JSON.stringify({side,out_pid:outP,in_pid:inP})\n"
    "  })).json();\n"
    "  const diff=Math.round((d.proj-d.cur)*100),up=diff>=0;\n"
    "  const col=up?'var(--green)':'var(--red)';\n"
    "  document.getElementById('sub-res').innerHTML=\n"
    "    '<div class=\"alert\" style=\"border-color:'+col+';color:'+col+';background:rgba(0,0,0,.2)\">'\n"
    "    +'Preview: '+Math.round(d.cur*100)+'% to <strong>'+Math.round(d.proj*100)+'%</strong>'\n"
    "    +' ('+(up?'up +':'down ')+Math.abs(diff)+'%)'\n"
    "    +'</div>';\n"
    "}\n"
    "\n"
    "async function applySub(){\n"
    "  const side=document.getElementById('sub-side').value;\n"
    "  const outP=parseInt(document.getElementById('sub-out').value);\n"
    "  const inP=parseInt(document.getElementById('sub-in').value);\n"
    "  if(!outP||!inP) return;\n"
    "  const d=await (await fetch('/api/apply_sub',{\n"
    "    method:'POST',headers:{'Content-Type':'application/json'},\n"
    "    body:JSON.stringify({side,out_pid:outP,in_pid:inP})\n"
    "  })).json();\n"
    "  PR[side+Number(outP)]='bench'; PR[side+Number(inP)]='starter';\n"
    "  [[outP,'bench'],[inP,'starter']].forEach(([pid,role])=>{\n"
    "    const card=document.getElementById('pc-'+side+'-'+pid);\n"
    "    const badge=document.getElementById('pr-'+side+'-'+pid);\n"
    "    if(card) card.className='pc '+role;\n"
    "    if(badge){badge.className='pr r'+(role==='starter'?'s':'b');badge.textContent=role.toUpperCase();}\n"
    "  });\n"
    "  updCounts(side); refreshSubs();\n"
    "  updProb(d.probA,d.features);\n"
    "  const R=side==='A'?RA:RB;\n"
    "  const pOut=R.all.find(x=>x.id===outP)||{name:'Player '+outP};\n"
    "  const pIn=R.all.find(x=>x.id===inP)||{name:'Player '+inP};\n"
    "  document.getElementById('sub-res').innerHTML=\n"
    "    '<div class=\"alert as\">Applied: '+pOut.name+' replaced by '+pIn.name+'. Prob: '+Math.round(d.probA*100)+'%</div>';\n"
    "  pushProb(d.probA,'Sub:'+pIn.name.split(' ')[0]);\n"
    "  buildBayDd();\n"
    "}\n"
    "\n"
    "function buildPerfGrid(){\n"
    "  const c=document.getElementById('perf-grid'); c.innerHTML='';\n"
    "  const tA=document.getElementById('dd_A').value,tB=document.getElementById('dd_B').value;\n"
    "  [['A',tA],['B',tB]].forEach(([side,tid])=>{\n"
    "    const R=side==='A'?RA:RB;\n"
    "    const div=document.createElement('div');\n"
    "    div.innerHTML='<div style=\"font-weight:700;color:'+(side==='A'?'var(--blue)':'var(--red)')+';margin-bottom:10px\">'+TM[tid]+'</div>';\n"
    "    R.all.forEach(p=>{\n"
    "      const row=document.createElement('div'); row.className='sr';\n"
    "      row.innerHTML='<span class=\"sl\" style=\"flex:1\">'+p.name+'</span>'\n"
    "        +'<input type=\"number\" id=\"pf-'+side+'-'+p.id+'\" value=\"0\" min=\"0\" max=\"40\"'\n"
    "        +' style=\"width:72px;padding:3px 7px;font-size:.8rem;background:#0d1117;'\n"
    "        +'color:var(--text);border:1px solid var(--border);border-radius:6px\">';\n"
    "      div.appendChild(row);\n"
    "    });\n"
    "    c.appendChild(div);\n"
    "  });\n"
    "}\n"
    "\n"
    "async function saveMatch(){\n"
    "  const tA=document.getElementById('dd_A').value,tB=document.getElementById('dd_B').value;\n"
    "  const sA=parseInt(document.getElementById('inp-A').value)||0;\n"
    "  const sB=parseInt(document.getElementById('inp-B').value)||0;\n"
    "  const perfs=[];\n"
    "  [['A',tA],['B',tB]].forEach(([side,tid])=>{\n"
    "    const R=side==='A'?RA:RB;\n"
    "    R.all.forEach(p=>{\n"
    "      const el=document.getElementById('pf-'+side+'-'+p.id);\n"
    "      const pts=el?parseFloat(el.value)||0:0;\n"
    "      perfs.push({pid:p.id,name:p.name,team_id:parseInt(tid),points:pts});\n"
    "    });\n"
    "  });\n"
    "  const winner=sA>sB?TM[tA]:(sB>sA?TM[tB]:'Tied');\n"
    "  const result=sA!==sB?(winner+' beat '+(sA>sB?TM[tB]:TM[tA])+' ('+sA+' - '+sB+')'):'Match Tied';\n"
    "  const d=await (await fetch('/api/save_match',{\n"
    "    method:'POST',headers:{'Content-Type':'application/json'},\n"
    "    body:JSON.stringify({result,player_performances:perfs,date:new Date().toLocaleDateString()})\n"
    "  })).json();\n"
    "  document.getElementById('save-msg').innerHTML=d.ok\n"
    "    ?'<div class=\"alert as\">Saved: '+d.msg+'</div>'\n"
    "    :'<div class=\"alert ad\">Warning: '+d.msg+'</div>';\n"
    "}\n"
    "\n"
    "async function loadH2H(){\n"
    "  const tA=document.getElementById('h2h-A').value,tB=document.getElementById('h2h-B').value;\n"
    "  const c=document.getElementById('h2h-content');\n"
    "  if(!tA||!tB){c.innerHTML='<div class=\"alert aw\">Select both teams.</div>';return;}\n"
    "  if(tA===tB){c.innerHTML='<div class=\"alert ad\">Choose different teams.</div>';return;}\n"
    "  c.innerHTML='<div class=\"alert ai\"><span class=\"ld\"></span> Loading...</div>';\n"
    "  const d=await (await fetch('/api/h2h?tidA='+tA+'&tidB='+tB)).json();\n"
    "  const recHtml=d.recent.map(r=>\n"
    "    '<tr><td><span class=\"badge '+(r.winner==='Tie'?'by':r.winner===d.tnA?'bg2':'br')+'\">'\n"
    "    +r.winner+'</span></td><td>'+r.result+'</td></tr>'\n"
    "  ).join('');\n"
    "  c.innerHTML=\n"
    "    '<div class=\"g3 mb14\">'\n"
    "    +'<div class=\"card\" style=\"text-align:center\"><h3>'+d.tnA+'</h3>'\n"
    "    +'<div style=\"font-size:3rem;font-weight:800;color:var(--green)\">'+d.wA+'</div>'\n"
    "    +'<div style=\"color:var(--muted);font-size:.85rem\">Wins</div></div>'\n"
    "    +'<div class=\"card\" style=\"text-align:center\"><h3>Ties</h3>'\n"
    "    +'<div style=\"font-size:3rem;font-weight:800;color:var(--yellow)\">'+d.ties+'</div></div>'\n"
    "    +'<div class=\"card\" style=\"text-align:center\"><h3>'+d.tnB+'</h3>'\n"
    "    +'<div style=\"font-size:3rem;font-weight:800;color:var(--red)\">'+d.wB+'</div>'\n"
    "    +'<div style=\"color:var(--muted);font-size:.85rem\">Wins</div></div>'\n"
    "    +'</div>'\n"
    "    +'<div class=\"card\"><h3>Last '+d.recent.length+' Matches Between These Teams</h3>'\n"
    "    +(d.recent.length\n"
    "      ?'<table><thead><tr><th>Winner</th><th>Result</th></tr></thead><tbody>'+recHtml+'</tbody></table>'\n"
    "      :'<div class=\"alert ai\">No head-to-head matches found in the dataset.</div>')\n"
    "    +'</div>';\n"
    "}\n"
    "\n"
    "function resetMatch(){\n"
    "  if(!confirm('Reset all match data?')) return;\n"
    "  fetch('/api/reset',{method:'POST'}).then(()=>{\n"
    "    PR={}; RA=RB={starters:[],bench:[],all:[]}; probHist=[];\n"
    "    if(probChart){probChart.destroy();probChart=null;}\n"
    "    ['dd_A','dd_B'].forEach(id=>document.getElementById(id).value='');\n"
    "    ['prevA','prevB','setup-msg'].forEach(id=>document.getElementById(id).innerHTML='');\n"
    "    document.getElementById('step1').className='step active';\n"
    "    ['step2','step3','step4'].forEach(id=>document.getElementById(id).className='step');\n"
    "    showTab('setup');\n"
    "  });\n"
    "}\n"
    "\n"
    "init();\n"
    "</script>\n"
    "</body></html>\n"
)

# --- Flask routes ---
@app.route('/')
def index(): return render_template_string(HTML_PAGE)

@app.route('/api/teams')
def api_teams():
    t = teams[['id','name']].drop_duplicates().dropna(subset=['name']).sort_values('name')
    return jsonify({str(int(r['id'])): str(r['name']) for _,r in t.iterrows()})

@app.route('/api/team_stats')
def api_team_stats():
    tid = int(request.args.get('tid'))
    return jsonify(get_team_stats(tid))

@app.route('/api/roster')
def api_roster():
    tidA=int(request.args.get('tidA')); tidB=int(request.args.get('tidB'))
    def build(tid):
        st,bn,al = roster_for_team(tid)
        return {'starters':[get_player_info(p) for p in st],
                'bench'   :[get_player_info(p) for p in bn],
                'all'     :[get_player_info(p) for p in al]}
    rA=build(tidA); rB=build(tidB)
    seen=set(); all_p=[]
    for p in rA['all']+rB['all']:
        if p['id'] not in seen: seen.add(p['id']); all_p.append(p)
    return jsonify({'rA':rA,'rB':rB,'all':all_p})

@app.route('/api/confirm_lineup',methods=['POST'])
def api_confirm_lineup():
    d=request.json
    match_state.update(teamA_id=int(d['tA']),teamB_id=int(d['tB']),
        starters_A=d['stA'],bench_A=d['bnA'],starters_B=d['stB'],bench_B=d['bnB'],
        scoreA=0,scoreB=0,features={},probA=0.5)
    compute_features(match_state); p=predict_win(match_state)
    return jsonify({'probA':p,'features':match_state['features']})

@app.route('/api/update_score',methods=['POST'])
def api_update_score():
    d=request.json
    match_state['scoreA']=int(d['scoreA']); match_state['scoreB']=int(d['scoreB'])
    compute_features(match_state); p=predict_win(match_state)
    return jsonify({'probA':p,'features':match_state['features']})

@app.route('/api/bayes_update',methods=['POST'])
def api_bayes_update():
    d=request.json; prior=match_state['probA']
    post=bayes_update(prior,d['pid'],float(d['observed']))
    match_state['probA']=post
    return jsonify({'probA':post,'prior':prior})

@app.route('/api/simulate_sub',methods=['POST'])
def api_simulate_sub():
    d=request.json
    proj=simulate_sub(match_state,d['side'],int(d['out_pid']),int(d['in_pid']))
    return jsonify({'proj':proj,'cur':match_state['probA']})

@app.route('/api/apply_sub',methods=['POST'])
def api_apply_sub():
    d=request.json; side,op,ip=d['side'],int(d['out_pid']),int(d['in_pid'])
    sk,bk='starters_'+side,'bench_'+side
    if op in match_state[sk]:    match_state[sk].remove(op)
    if ip not in match_state[sk]: match_state[sk].append(ip)
    if ip  in match_state[bk]:   match_state[bk].remove(ip)
    if op not in match_state[bk]: match_state[bk].append(op)
    compute_features(match_state); p=predict_win(match_state)
    return jsonify({'probA':p,'features':match_state['features']})

@app.route('/api/squad_stats')
def api_squad_stats():
    return jsonify(get_squad_stats(int(request.args.get('tid'))))

@app.route('/api/h2h')
def api_h2h():
    return jsonify(get_h2h(int(request.args.get('tidA')),int(request.args.get('tidB'))))

@app.route('/api/save_match',methods=['POST'])
def api_save_match():
    d=request.json
    d['match_id']=int(matches['match_id'].max())+1 if not matches.empty else 9001
    ok,msg=save_match_to_db(d)
    return jsonify({'ok':ok,'msg':msg})

@app.route('/api/reset',methods=['POST'])
def api_reset():
    match_state.update(teamA_id=None,teamB_id=None,
        starters_A=[],bench_A=[],starters_B=[],bench_B=[],
        scoreA=0,scoreB=0,features={},probA=0.5)
    return jsonify({'ok':True})

# --- Launch ---
_url = f'http://127.0.0.1:{PORT}'
display(HTML(
    "<div style='background:#161b22;border:1px solid #30363d;border-radius:12px;"
    "padding:22px;max-width:500px;font-family:system-ui'>"
    "<div style='font-size:1.2rem;font-weight:700;color:#58a6ff;margin-bottom:8px'>"
    "Dashboard is Live!</div>"
    "<div style='color:#7d8590;margin-bottom:14px;font-size:.9rem'>"
    "Flask server running in background thread. Notebook stays interactive.</div>"
    f"<a href='{_url}' target='_blank'"
    " style='display:inline-block;background:#58a6ff;color:#000;padding:11px 22px;"
    f"border-radius:8px;text-decoration:none;font-weight:700'>Open Dashboard: {_url}</a>"
    "<div style='margin-top:14px;color:#3fb950;font-size:.82rem'>"
    
    "</div></div>"
))

def _open():
    time.sleep(1.2)
    webbrowser.open(_url)

threading.Thread(target=_open, daemon=True).start()
threading.Thread(
    target=lambda: app.run(host='127.0.0.1', port=PORT, debug=False, use_reloader=False),
    daemon=True
).start()
print(f'Server: {_url}')


Server: http://127.0.0.1:5055
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5055
Press CTRL+C to quit
